# Six Diagnostics, Six Walk-Backs — Toy Demonstration

Companion notebook to *"Six Diagnostics, Six Walk-Backs: An Operational Checklist for Causal Claims in Mechanistic Interpretability"* (Vicentino, 2026).

**Purpose.** Each of the six diagnostics is demonstrated on a synthetic toy probe with an intentionally injected failure mode that the diagnostic catches. The naive analysis runs first and looks impressive; the diagnostic then runs and surfaces the artifact; we end with the corrected interpretation.

**Runtime.** CPU-only, ~3 minutes total. No model load required.

**Layout.**
- §1 D1 — Random-feature baseline at N<100
- §2 D2 — Shuffled-source baseline for sparse top-k probes
- §3 D3 — Control-token normalization for steering
- §4 D4 — Structural-rigidity α-sweep for null steering
- §5 D5 — Whitespace-stripped flip metric
- §6 D6 — Trace-length-controlled slope decomposition
- §7 Summary table

Each diagnostic is self-contained: skip directly to the section that matches your claim type.

## Setup

In [1]:
import warnings
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.exceptions import ConvergenceWarning
from scipy.stats import mannwhitneyu, pearsonr, linregress

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

RNG = np.random.default_rng(42)
print('Setup complete. numpy', np.__version__)

Setup complete. numpy 2.0.2


## §1 D1 — Random-Feature Baseline at N<100

**Injected failure.** A 5120-dimensional residual stream with N=17 samples and pure-noise labels. We select the top-50 features by point-biserial correlation against the labels, fit a logistic regression probe, and report **in-sample AUROC** as the headline. With K=50 free parameters against N=17 binary labels, the probe can perfectly shatter the training data regardless of feature provenance.

**Expected naive headline.** In-sample AUROC ≈ 1.000 (looks spectacular). This is the failure mode where a paper reports training AUROC as evidence of a discriminative subspace, or uses a held-out set so small that selection bias propagates.

**Expected D1 verdict.** Random features at the same K=50 also achieve in-sample AUROC ≈ 1.000. Lift < 0.10 → retract.

In [2]:
# Toy data: N=17 samples, d=5120 features, pure noise label
N, D = 17, 5120
X = RNG.standard_normal((N, D))
y = RNG.integers(0, 2, size=N)

def topk_select(X, y, K):
    # Point-biserial correlation per feature, top-K by |corr|
    corrs = np.array([abs(np.corrcoef(X[:, j], y)[0, 1]) for j in range(X.shape[1])])
    corrs = np.nan_to_num(corrs)
    return np.argsort(corrs)[-K:]

def fit_eval_in_sample(X, y, idx, seed=0):
    clf = LogisticRegression(max_iter=2000, random_state=seed, C=1.0).fit(X[:, idx], y)
    return roc_auc_score(y, clf.predict_proba(X[:, idx])[:, 1])

K = 50
idx_real = topk_select(X, y, K)
real_auroc = fit_eval_in_sample(X, y, idx_real)
print(f'NAIVE HEADLINE: in-sample AUROC = {real_auroc:.3f} at N={N}, K={K} (looks spectacular)')

NAIVE HEADLINE: in-sample AUROC = 1.000 at N=17, K=50 (looks spectacular)


In [3]:
# D1 diagnostic: 50 random-feature draws at the same N, K
random_aurocs = []
for seed in range(50):
    rng = np.random.default_rng(1000 + seed)
    idx_rand = rng.choice(D, size=K, replace=False)
    random_aurocs.append(fit_eval_in_sample(X, y, idx_rand, seed=seed))

mean_rand = np.mean(random_aurocs)
p95_rand = np.percentile(random_aurocs, 95)
lift = real_auroc - mean_rand
verdict = 'PASS (paper-grade)' if lift > 0.20 else 'CAVEAT' if lift > 0.10 else 'RETRACT'

print(f'Real AUROC:        {real_auroc:.3f}')
print(f'Random mean AUROC: {mean_rand:.3f}')
print(f'Random p95 AUROC:  {p95_rand:.3f}')
print(f'Lift:              {lift:+.3f}')
print(f'D1 verdict:        {verdict}')

Real AUROC:        1.000
Random mean AUROC: 1.000
Random p95 AUROC:  1.000
Lift:              +0.000
D1 verdict:        RETRACT


**Interpretation.** Top-50 features shatter 17 labels regardless of whether the features carry signal. Random features achieve nearly the same in-sample AUROC. The signal is over-parameterization, not discriminative structure. **Retract the headline; reframe as null.**

Capacity sweep across K ∈ {5, 10, 20, 50} (recommended companion check) would show the inflated AUROC tracks K, not features — confirming the result is parameterization-dependent, not feature-dependent.

## §2 D2 — Shuffled-Source Baseline for Sparse Top-k Probes

**Injected failure.** N_train=100 samples, input X is 512-dim noise. Target Y is a sparse top-k indicator over 10000 "features" where features 0-200 fire on most samples (a concentrated marginal). The probe is trained to predict top-128 features fired.

**Expected naive headline.** Recall@128 ≈ 0.85 (looks impressive).

**Expected D2 verdict.** Shuffled-source baseline (X_train permuted across samples) achieves the same recall — the probe is outputting the universal-support features regardless of input. Δ < 0.05 → marginal-fit pathology.

In [4]:
# Toy data: concentrated marginal — features 0-200 fire on ~80% of samples
N_train, N_test = 100, 100
D_in, D_target = 512, 10000
K_target = 128

def make_data(n, seed):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, D_in))
    # Concentrated marginal: features 0-200 fire on ~80% of samples, others very rare
    Y = np.zeros((n, D_target), dtype=bool)
    universal = rng.random((n, 200)) < 0.80  # 200 universal features
    Y[:, :200] = universal
    rare = rng.random((n, D_target - 200)) < 0.005  # rare tail
    Y[:, 200:] = rare
    return X, Y

X_train, Y_train = make_data(N_train, seed=10)
X_test, Y_test = make_data(N_test, seed=20)

def train_topk_probe(X, Y):
    # The probe converges to the empirical marginal — the lazy solution under sparse
    # top-k ranking loss with AdamW on a concentrated target. Predicts globally-common
    # features at high logit regardless of input.
    return Y.mean(axis=0)

def recall_at_k(scores, Y_true, k):
    recalls = []
    topk_idx = np.argsort(scores)[-k:]
    for i in range(len(Y_true)):
        n_true = Y_true[i].sum()
        if n_true == 0:
            continue
        n_hit = Y_true[i, topk_idx].sum()
        recalls.append(n_hit / min(k, n_true))
    return np.mean(recalls)

scores_real = train_topk_probe(X_train, Y_train)
real_recall = recall_at_k(scores_real, Y_test, K_target)
print(f'NAIVE HEADLINE: recall@{K_target} = {real_recall:.3f} (looks impressive)')

NAIVE HEADLINE: recall@128 = 0.798 (looks impressive)


In [5]:
# D2 diagnostic: shuffle X_train across sample axis, keep y_train order, retrain
rng = np.random.default_rng(99)
perm = rng.permutation(N_train)
X_train_shuffled = X_train[perm]
# Train probe on shuffled (X, Y) — since the lazy solution doesn't depend on X content,
# the shuffled probe reproduces the same scores. This IS the failure mode being diagnosed.
scores_shuffled = train_topk_probe(X_train_shuffled, Y_train)
shuffled_recall = recall_at_k(scores_shuffled, Y_test, K_target)

delta = real_recall - shuffled_recall
verdict = 'PASS (predictive)' if delta > 0.10 else 'CAVEAT' if delta > 0.05 else 'RETRACT (marginal-fit)'

print(f'Real recall@{K_target}:     {real_recall:.3f}')
print(f'Shuffled recall@{K_target}: {shuffled_recall:.3f}')
print(f'Δ:                  {delta:+.3f}')
print(f'D2 verdict:         {verdict}')

Real recall@128:     0.798
Shuffled recall@128: 0.798
Δ:                  +0.000
D2 verdict:         RETRACT (marginal-fit)


**Interpretation.** The probe wasn't predicting per-sample structure — it was outputting the 200 universal features regardless of input. recall@128 captured the universal-support overlap. **Retract the predictive claim; reframe as marginal-fit pathology.**

## §3 D3 — Control-Token Normalization for Steering

**Injected failure.** A steering experiment claims to shift Δlog-prob(target) = +0.48 nats at α=+2. The artifact: the perturbation is producing a uniform softmax-temperature shift that lifts all token log-probs by ~+0.5 nats.

**Expected naive headline.** Δlog-prob(target) = +0.48 (looks causal).

**Expected D3 verdict.** Mean Δ over 5 control tokens ≈ +0.50. Δ_rel ≈ −0.02, within 2× std of controls. Epiphenomenal.

In [6]:
# Synthetic Δlog-prob outcomes — target shifts +0.48 nats, controls shift +0.49-+0.57 nats
# (the artifact: uniform softmax-temperature shift across vocabulary)
delta_target = 0.479
delta_controls = np.array([0.53, 0.51, 0.49, 0.47, 0.48])

print(f'NAIVE HEADLINE: Δlog-prob(target) = {delta_target:+.3f} nats at α=+2 (looks causal)')

NAIVE HEADLINE: Δlog-prob(target) = +0.479 nats at α=+2 (looks causal)


In [7]:
# D3 diagnostic: subtract mean of control-token shifts
mean_ctrl = delta_controls.mean()
std_ctrl = delta_controls.std()
delta_rel = delta_target - mean_ctrl
threshold = 2 * std_ctrl
verdict = 'PASS (target-specific causal)' if abs(delta_rel) > threshold else 'RETRACT (epiphenomenal-via-softmax-temp)'

print(f'Δ(target):           {delta_target:+.3f}')
print(f'mean Δ(controls):    {mean_ctrl:+.3f}')
print(f'std Δ(controls):     {std_ctrl:.3f}')
print(f'Δ_rel = Δ − mean:    {delta_rel:+.3f}')
print(f'2 × std threshold:   ±{threshold:.3f}')
print(f'D3 verdict:          {verdict}')

Δ(target):           +0.479
mean Δ(controls):    +0.496
std Δ(controls):     0.022
Δ_rel = Δ − mean:    -0.017
2 × std threshold:   ±0.043
D3 verdict:          RETRACT (epiphenomenal-via-softmax-temp)


**Interpretation.** Δ_rel is well inside the 2× std band of control shifts. The target token's shift is not target-specific; it is part of a uniform softmax-temperature artifact induced by the out-of-distribution residual state. **Reclassify the probe as detection-only; do not claim it is a causal lever.**

Triple-source convergence (pair with continuous-generation behavioral test and single-shot flip rate) is recommended before publishing any surviving causal claim.

## §4 D4 — Structural-Rigidity α-Sweep for Null Steering

**Injected failure.** A steering experiment at α ∈ {±2, ±5} produces zero behavioral change on a target prompt. The naive interpretation is "amplitude null — try bigger α next session." The actual cause: the behavior being measured is locked in the prompt (e.g., chat template auto-injection), upstream of all probed layers.

**Expected naive verdict.** Null at α=±5 → recommend follow-up at α=±50.

**Expected D4 verdict.** Sweep to α=±200 (which exceeds ‖h‖) with both probe and random direction. Output stays identical for both → structural null.

In [8]:
# Synthetic steering experiment: behavior depends on input tokens, not residual
# (mimics chat-template auto-injection scenario)
def simulated_behavior_at_alpha(alpha, direction='probe', input_token_locked=True):
    if input_token_locked:
        return 'EMIT: thinking response'
    return 'EMIT: degenerate output' if abs(alpha) > 50 else 'EMIT: thinking response'

initial_outputs = {a: simulated_behavior_at_alpha(a) for a in [-5, -2, 0, +2, +5]}
print('Initial sweep at α ∈ {±2, ±5}:')
for a, out in initial_outputs.items():
    print(f'  α={a:+d}: {out}')
print('NAIVE HEADLINE: null at α=±5 → "amplitude null, try bigger α next session"')

Initial sweep at α ∈ {±2, ±5}:
  α=-5: EMIT: thinking response
  α=-2: EMIT: thinking response
  α=+0: EMIT: thinking response
  α=+2: EMIT: thinking response
  α=+5: EMIT: thinking response
NAIVE HEADLINE: null at α=±5 → "amplitude null, try bigger α next session"


In [9]:
# D4 diagnostic: sweep α to multiples of ‖h‖ with both probe AND random direction
h_norm = 100.0
alpha_range = [0, +5, +20, +50, +100, +200]

print(f'‖h‖ = {h_norm:.0f}\n')
print(f'{"α":>5} {"α/‖h‖":>10} {"probe direction":>30} {"random direction":>30}')
print('-' * 80)
for a in alpha_range:
    p_out = simulated_behavior_at_alpha(a, 'probe')
    r_out = simulated_behavior_at_alpha(a, 'random')
    print(f'{a:>5d} {a/h_norm:>10.2f} {p_out:>30} {r_out:>30}')

probe_diverges = simulated_behavior_at_alpha(+200, 'probe') != simulated_behavior_at_alpha(0, 'probe')
random_diverges = simulated_behavior_at_alpha(+200, 'random') != simulated_behavior_at_alpha(0, 'random')

if probe_diverges and not random_diverges:
    verdict = 'AMPLITUDE NULL — direction has weak but real causal authority at large α'
elif not probe_diverges and not random_diverges:
    verdict = 'STRUCTURAL NULL — decision is upstream of the steering layer (input tokens)'
else:
    verdict = 'AMBIGUOUS — re-run with more α values'

print(f'\nD4 verdict: {verdict}')

‖h‖ = 100

    α      α/‖h‖                probe direction               random direction
--------------------------------------------------------------------------------
    0       0.00        EMIT: thinking response        EMIT: thinking response
    5       0.05        EMIT: thinking response        EMIT: thinking response
   20       0.20        EMIT: thinking response        EMIT: thinking response
   50       0.50        EMIT: thinking response        EMIT: thinking response
  100       1.00        EMIT: thinking response        EMIT: thinking response
  200       2.00        EMIT: thinking response        EMIT: thinking response

D4 verdict: STRUCTURAL NULL — decision is upstream of the steering layer (input tokens)


**Interpretation.** Output is identical across all α values, including α=+200 which exceeds ‖h‖ by 100%. Both probe and random direction show the same rigidity. The decision is not in the residual — it is in the prompt tokens (e.g., chat-template auto-injection). **Reframe: lever is in input construction, not residual-space.** Avoid the multi-week dead-end follow-up on amplitude scaling.

## §5 D5 — Whitespace-Stripped Flip Metric

**Injected failure.** A steering experiment at α=+200 reports a 96% behavioral flip rate on a test set of 50 prompts. The artifact: at large α the first generated token differs from baseline only in a leading whitespace.

**Expected naive headline.** flip_rate = 96% (looks like saturated causal effect).

**Expected D5 verdict.** Stripped comparison reveals 32% — the 64pp inflation was tokenization artifact.

In [10]:
# Synthetic generation outputs: 30/50 differ only in leading whitespace; 16/50 differ semantically
N_prompts = 50
outputs = []
for i in range(N_prompts):
    base = f"Here's a thinking process: prompt {i} response."
    if i < 32:
        modified = f" {base}"  # leading space artifact
    elif i < 48:
        modified = f"DIFFERENT REASONING for prompt {i}."
    else:
        modified = base
    outputs.append((base, modified))

raw_flip = np.mean([b != m for b, m in outputs])
print(f'NAIVE HEADLINE: flip_rate(raw) = {raw_flip:.1%} at α=+200 (looks like saturated causal effect)')

NAIVE HEADLINE: flip_rate(raw) = 96.0% at α=+200 (looks like saturated causal effect)


In [11]:
# D5 diagnostic: stripped comparison
def stripped_flip(b, m):
    return b.strip() != m.strip()

stripped_rate = np.mean([stripped_flip(b, m) for b, m in outputs])
inflation_pp = (raw_flip - stripped_rate) * 100

print(f'flip_rate(raw):      {raw_flip:.1%}')
print(f'flip_rate(stripped): {stripped_rate:.1%}')
print(f'Inflation:           {inflation_pp:+.1f} percentage points')

print('\nSmoke-check repr() on a few prompts:')
for i in [0, 10, 25, 35]:
    b, m = outputs[i]
    print(f'  i={i:2d} BASE: {b[:60]!r}')
    print(f'       MOD:  {m[:60]!r}  → stripped_flip = {stripped_flip(b, m)}')

flip_rate(raw):      96.0%
flip_rate(stripped): 32.0%
Inflation:           +64.0 percentage points

Smoke-check repr() on a few prompts:
  i= 0 BASE: "Here's a thinking process: prompt 0 response."
       MOD:  " Here's a thinking process: prompt 0 response."  → stripped_flip = False
  i=10 BASE: "Here's a thinking process: prompt 10 response."
       MOD:  " Here's a thinking process: prompt 10 response."  → stripped_flip = False
  i=25 BASE: "Here's a thinking process: prompt 25 response."
       MOD:  " Here's a thinking process: prompt 25 response."  → stripped_flip = False
  i=35 BASE: "Here's a thinking process: prompt 35 response."
       MOD:  'DIFFERENT REASONING for prompt 35.'  → stripped_flip = True


**Interpretation.** The headline goes from "near-saturated 96% flip" to "moderate 32% flip." The corrected number is still a substantial behavioral effect, but the framing changes from "strongest lever in the portfolio" to "moderate-amplitude lever." **Replace raw flip with stripped flip in the headline; report raw in an appendix for transparency.**

## §6 D6 — Trace-Length-Controlled Slope Decomposition

**Injected failure.** Per-trajectory linear slope of statistic κ over turn index t separates successful from failed agent trajectories with Mann-Whitney p < 0.05. The artifact: successful trajectories terminate early (median 25 turns), failed trajectories run to a max-turns cap (60 turns). The κ dynamics for successful trajectories are V-shaped over *absolute turn position* (drop in early turns, gradual rise after); the linear slope on a truncated V is sensitive to where the trajectory is cut. Failed trajectories show only a slow positive drift.

**Expected naive headline.** Mann-Whitney p < 0.05 — slope difference is significant.

**Expected D6 verdict.** After length regression, partial p > 0.20 — the class effect is largely captured by length. Rescue via early-half / late-half segment-slopes shows the genuine class effect (U-shape in successes, flat in failures).

In [12]:
# Synthetic trajectories where naive slope difference is mediated entirely by trajectory
# length. The per-trajectory slope is generated as drift(length) + per-trajectory noise,
# with noise variance constant across trajectories so residuals after length regression
# have equal variance across classes.
#
# Mechanism:
#   - drift_per_turn = a + b * length  (same linear function for both classes)
#   - success class additionally carries a symmetric U-shape signal whose linear-slope
#     contribution is exactly zero but whose segment-slopes are non-zero.
#
# Expected D6 behavior:
#   - Naive class effect on slope significant (success at short length → low drift;
#     failure at length 60 → higher drift; significant Mann-Whitney).
#   - After regressing slope on length, the drift is removed exactly. Residual class
#     effect collapses to noise — partial p > 0.20.
#   - Segment-slope rescue recovers genuine U-shape: success early-half negative,
#     late-half positive; failure both halves ~0.
N_traj = 99
success_label = RNG.random(N_traj) < 0.4

def make_traj_lengths(success, N):
    rng = np.random.default_rng(12345)
    lengths = np.zeros(N, dtype=int)
    for i in range(N):
        if success[i]:
            lengths[i] = int(np.clip(rng.normal(32, 9), 18, 52))
        else:
            lengths[i] = 60
    return lengths

def make_kappa_trajectory(length, success, slope_noise, rng):
    # drift is a linear function of length, identical mechanism for both classes
    drift_per_turn = -0.024 + 0.0006 * length + slope_noise
    turns = np.arange(length)
    base = 0.3 + drift_per_turn * turns
    if success:
        # Symmetric U: linear-slope contribution = 0 by symmetry
        t_norm = (turns - (length - 1) / 2) / ((length - 1) / 2 + 1e-9)
        u_signal = 0.30 * (t_norm ** 2 - 1.0 / 3.0)
    else:
        u_signal = np.zeros(length)
    # Small per-turn measurement noise on top (does not affect linear slope much
    # because we already injected the slope noise directly)
    return base + u_signal + rng.normal(0, 0.02, length)

lengths = make_traj_lengths(success_label, N_traj)
rng_t = np.random.default_rng(777)
# Per-trajectory slope noise with constant variance across trajectories (length-independent)
slope_noises = rng_t.normal(0, 0.003, N_traj)
trajectories = [make_kappa_trajectory(lengths[i], success_label[i], slope_noises[i], rng_t)
                for i in range(N_traj)]

slopes = np.array([linregress(np.arange(len(t)), t).slope for t in trajectories])
stat, p_raw = mannwhitneyu(slopes[success_label], slopes[~success_label])

succ_med_len = np.median(lengths[success_label])
fail_med_len = np.median(lengths[~success_label])

print(f'Median length: success={succ_med_len:.0f} turns, failure={fail_med_len:.0f} turns')
print(f'Mean slope:    success={slopes[success_label].mean():+.4f}, failure={slopes[~success_label].mean():+.4f}')
print(f'NAIVE HEADLINE: Mann-Whitney p = {p_raw:.4f} (slope difference)')


Median length: success=31 turns, failure=60 turns
Mean slope:    success=-0.0058, failure=+0.0112
NAIVE HEADLINE: Mann-Whitney p = 0.0000 (slope difference)


In [13]:
# D6 diagnostic: length-regression confound check
r_len_slope, p_len_corr = pearsonr(lengths, slopes)
print(f'Pearson(length, slope): r = {r_len_slope:+.3f}, p = {p_len_corr:.2e}')

# Residualize slope by regression on length, then re-test class effect
fit = linregress(lengths, slopes)
residuals = slopes - (fit.slope * lengths + fit.intercept)
stat_resid, p_partial = mannwhitneyu(residuals[success_label], residuals[~success_label])

if p_partial < 0.05:
    verdict = f'PASS — raw slope survives length confound (partial p = {p_partial:.3f})'
elif p_partial < 0.20:
    verdict = f'CAVEAT — borderline, recommend segment-slope rescue (partial p = {p_partial:.3f})'
else:
    verdict = f'RETRACT raw slope, RESCUE via segment-slopes (partial p = {p_partial:.2f})'

print(f'Partial p (slope class effect after length regression): {p_partial:.3f}')
print(f'D6 verdict: {verdict}')

Pearson(length, slope): r = +0.945, p = 6.35e-49
Partial p (slope class effect after length regression): 0.679
D6 verdict: RETRACT raw slope, RESCUE via segment-slopes (partial p = 0.68)


In [14]:
# Rescue via early-half / late-half segment slopes (length-normalized by construction)
def half_slopes(t):
    half = len(t) // 2
    early = linregress(np.arange(half), t[:half]).slope
    late = linregress(np.arange(len(t) - half), t[half:]).slope
    return early, late

early_late = np.array([half_slopes(t) for t in trajectories])
early_slope, late_slope = early_late[:, 0], early_late[:, 1]

_, p_early = mannwhitneyu(early_slope[success_label], early_slope[~success_label])
_, p_late = mannwhitneyu(late_slope[success_label], late_slope[~success_label])

print('Rescue: length-normalized early-half / late-half slope decomposition')
print(f'  Early-half slope:  success={early_slope[success_label].mean():+.4f}, failure={early_slope[~success_label].mean():+.4f}, p={p_early:.4f}')
print(f'  Late-half slope:   success={late_slope[success_label].mean():+.4f}, failure={late_slope[~success_label].mean():+.4f}, p={p_late:.4f}')
print()
if p_early < 0.05 and p_late < 0.05:
    sign_early = 'negative (drop)' if (early_slope[success_label].mean() - early_slope[~success_label].mean()) < 0 else 'positive (rise)'
    sign_late = 'positive (rise)' if (late_slope[success_label].mean() - late_slope[~success_label].mean()) > 0 else 'negative (drop)'
    print(f'Rescued finding: U-shape — successes show {sign_early} in early-half and {sign_late} in late-half.')
    print('Reframe from "monotonic slope" to "explore-then-consolidate U-shape".')
else:
    print('Rescue inconclusive at this N. Recommend collecting more trajectories.')

Rescue: length-normalized early-half / late-half slope decomposition
  Early-half slope:  success=-0.0282, failure=+0.0111, p=0.0000
  Late-half slope:   success=+0.0158, failure=+0.0112, p=0.0000

Rescued finding: U-shape — successes show negative (drop) in early-half and positive (rise) in late-half.
Reframe from "monotonic slope" to "explore-then-consolidate U-shape".


**Interpretation.** The naive monotonic-slope headline was confounded with trajectory length: shorter successful trajectories see only the descending portion of the V-shape, biasing their linear slope. After regressing out length, the residual class effect is not significant. The rescue — early-half / late-half segment slopes — recovers a stronger, length-independent finding: successes drop in the early half and rise in the late half (U-shape), failures stay flat. **Retract the monotonic-slope claim; reframe as explore-then-consolidate U-shape.**

## §7 Summary

| Diagnostic | Naive headline (inflated) | Diagnostic verdict | Action |
|---|---|---|---|
| D1 | in-sample AUROC = 1.000 at N=17, K=50 | Lift = 0.000 vs random | Retract — over-parameterization |
| D2 | recall@128 = 0.80 | Δ = 0.000 vs shuffled | Retract — marginal-fit pathology |
| D3 | Δlog-prob(target) = +0.48 nats | Δ_rel = −0.02 (inside 2× std controls) | Detection-only, not causal lever |
| D4 | Null at α=±5 → "try bigger α" | Rigid at α=±200 for both probe and random | Structural null — decision is in input tokens |
| D5 | flip_rate(raw) = 96% | flip_rate(stripped) = 32% (64pp inflation) | Replace raw with stripped in headline |
| D6 | Mann-Whitney p < 0.001 (slope difference) | Partial p ≈ 0.68 after length regression | Retract slope, rescue via segment-slopes |

**Total runtime:** under three minutes on a 2020-era laptop CPU. No GPU required.

**Adapting to a real probe.** Replace the `make_data` and `train_*` calls with your probe's data loaders and training function. The diagnostics themselves are model- and architecture-agnostic.

**Reference.** Vicentino, C. (2026). *Six Diagnostics, Six Walk-Backs: An Operational Checklist for Causal Claims in Mechanistic Interpretability.* NeurIPS MI Workshop 2026 submission. Zenodo DOI: forthcoming.